In [1]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("all-mpnet-base-v2")

/home/sudip/Desktop/agent_murdock/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3356.85it/s]


In [2]:
faiss_text = "What is FAISS. FAISS is a local vector library created by Facebook AI Research for fast matrix math."
weaviate_text = "Explaining Vector Databases. Weaviate is an AI-native database that stores both objects and vectors."

In [3]:
v1 = model.encode(weaviate_text).tolist()

In [4]:
v2 = model.encode(faiss_text).tolist()

In [13]:
query_vector = model.encode("Meta's open source embedding search tool").tolist()

In [6]:
import os
import weaviate
from dotenv import load_dotenv
from weaviate.classes.config import Configure, DataType, Property
load_dotenv()
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

with weaviate.connect_to_embedded(
    headers={"X-Goog-Api-Key": GEMINI_API_KEY}
) as client:
    collection_name = "Articles"

    if client.collections.exists(collection_name):
        client.collections.delete(collection_name)
    
    articles = client.collections.create(
        name=collection_name,
        vectorizer_config=Configure.Vectorizer.none(),
        properties=[
            Property(name='title', data_type=DataType.TEXT),
            Property(name='body', data_type=DataType.TEXT),
        ]
    )

    articles.data.insert(
        properties={
            "title": "Explaining Vector Databases",
            "body": "Weaviate is an AI-native database that stores both objects and vectors."
        },
        vector=v1
    )
    articles.data.insert(
        properties={
            "title": "What is FAISS",
            "body": "FAISS is a local vector library created by Facebook AI Research for fast matrix math."
        },
        vector=v2
    )

{"build_git_commit":"","build_go_version":"go1.24.3","build_image_tag":"","build_wv_version":"1.30.5","level":"warning","log_level_env":"","msg":"log level not recognized, defaulting to info","time":"2026-05-18T19:15:54+05:45"}
{"action":"startup","build_git_commit":"","build_go_version":"go1.24.3","build_image_tag":"","build_wv_version":"1.30.5","level":"info","msg":"Feature flag LD integration disabled: could not locate WEAVIATE_LD_API_KEY env variable","time":"2026-05-18T19:15:54+05:45"}
{"action":"startup","build_git_commit":"","build_go_version":"go1.24.3","build_image_tag":"","build_wv_version":"1.30.5","default_vectorizer_module":"none","level":"info","msg":"the default vectorizer modules is set to \"none\", as a result all new schema classes without an explicit vectorizer setting, will use this vectorizer","time":"2026-05-18T19:15:54+05:45"}
{"action":"startup","auto_schema_enabled":{},"build_git_commit":"","build_go_version":"go1.24.3","build_image_tag":"","build_wv_version":"

In [7]:
from numpy import dot
from numpy.linalg import norm

cos_sim = lambda a, b: dot(a, b) / (norm(a) * norm(b))

In [14]:
print("Similarity to Weaviate article:", cos_sim(query_vector, v1))
print("Similarity to FAISS article:", cos_sim(query_vector, v2))

Similarity to Weaviate article: 0.24015894974673396
Similarity to FAISS article: 0.21078403235477725
